# LC 84 — Largest Rectangle in Histogram
**Difficulty:** Hard | **Category:** Monotonic Stack
**Pattern:** Increasing Monotonic Stack (Area Expansion)

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Each bar is the shortest bar of some
rectangle. Use an increasing stack: when a shorter bar arrives, the
bar we pop is the height; the width extends from the new stack top
to the current index. Append a sentinel 0 to flush all remaining bars.
</div>

## Official Problem Statement

Given an array of integers `heights` representing the histogram's bar
heights where the width of each bar is `1`, return the area of the
largest rectangle in the histogram.

**Constraints:**
- `1 <= heights.length <= 100_000`
- `0 <= heights[i] <= 10_000`

## What This Is Actually Asking

You have a bar chart where each bar has width 1.
You want to find the biggest solid rectangle you can draw inside it.
The rectangle must be continuous (no gaps) and bounded by bar heights.
The limiting factor for any rectangle's height is its shortest bar.
Return the maximum area (height × width) of any such rectangle.

## Walk Through an Example by Hand

```
heights = [2, 1, 5, 6, 2, 3]  + sentinel [0]
indices    [0, 1, 2, 3, 4, 5]             [6]

stack=[] max_area=0

i=0, h=2: stack empty → push 0.  stack=[0]
i=1, h=1: 1<heights[0]=2 → pop 0
          width = 1-(-1)-1 = 1 (stack empty → left=-1)
          area = 2*1 = 2.  max=2
          push 1.  stack=[1]
i=2, h=5: 5>heights[1]=1 → push 2.  stack=[1,2]
i=3, h=6: 6>heights[2]=5 → push 3.  stack=[1,2,3]
i=4, h=2: 2<heights[3]=6 → pop 3
          width = 4-2-1 = 1. area=6*1=6. max=6
          2<heights[2]=5 → pop 2
          width = 4-1-1 = 2. area=5*2=10. max=10
          2>heights[1]=1 → push 4.  stack=[1,4]
i=5, h=3: 3>heights[4]=2 → push 5.  stack=[1,4,5]
i=6, h=0 (sentinel):
          0<heights[5]=3 → pop 5
          width=6-4-1=1. area=3*1=3
          0<heights[4]=2 → pop 4
          width=6-1-1=4. area=2*4=8
          0<heights[1]=1 → pop 1
          width=6-(-1)-1=6. area=1*6=6
          stack empty

Answer: 10
```

## The Picture

```
heights = [2, 1, 5, 6, 2, 3]

  6 |            ██
  5 |         ██ ██
  4 |         ██ ██
  3 |         ██ ██       ██
  2 |  ██     ██ ██ ██    ██
  1 |  ██ ██  ██ ██ ██    ██
       [0] [1] [2] [3] [4] [5]
        2   1   5   6   2   3

Largest rectangle (area=10): height=5, spans [2..3]

  5 |         ████████
  5 |         ████████
       [0] [1] [2] [3] [4] [5]

Stack state (stores indices, shown as heights):
  After i=1 pop: popped h=2, width=1  → area 2
  After i=4 pop: popped h=6, width=1  → area 6
                 popped h=5, width=2  → area 10 ← MAX
  Sentinel flushes remaining bars.

Width formula when popping idx p at position i:
  left_boundary = stack[-1] if stack else -1
  width = i - left_boundary - 1
```

## When To Use This Pattern

- When you see "largest rectangle" or "maximum area" in a histogram,
  think increasing monotonic stack.
- When a shorter bar "blocks" the expansion of taller bars to its
  right, think: pop and compute on block.
- When you need the nearest smaller bar to the left AND right
  simultaneously, think one-pass stack with width formula.
- When you want to flush all remaining candidates at the end, think
  appending a sentinel value of 0.
- When width depends on the element below the popped item in the
  stack, think "left boundary = new stack top".

## The Approach

Append a 0 sentinel to heights to flush the stack at the end.
Maintain an increasing stack of indices.
When a bar shorter than the stack top arrives, pop the top: that
bar's height is the rectangle height; width spans from the new
stack top (left boundary) to the current index (right boundary).
Track the maximum area seen across all pops.

In [12]:
from typing import List  # type hints for function signatures

In [13]:
def test_harness(func):
    """Run test cases for largestRectangleArea."""
    tests = [
        # (heights, expected_area)
        ([2,1,5,6,2,3], 10),
        ([2,4],          4),
        ([1],            1),
        ([0,0,0],        0),
        ([6,6,6,6],     24),
        ([1,2,3,4,5],   9),
    ]
    passed = 0
    for i, (heights, expected) in enumerate(tests):
        result = func(heights)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Input:    {heights}")
            print(f"  Expected: {expected}")
            print(f"  Got:      {result}")
    print(f"\n{passed}/{len(tests)} tests passed.")

In [14]:
r'''
For LC 84 (Largest Rectangle) — Increasing:

Stack holds indices of bars that are still "in play" for extending a rectangle rightward
When current bar is shorter than the top, that taller bar can't extend further — pop and calculate
Stack stays increasing bottom→top — taller bars sit on top until something shorter cuts them off
LC 84 is a monotonic increasing stack.
Bottom→top the indices stored represent bars of increasing height.
Processing is in two passes.
Pass A left to right
Pass B completely evict stack 
'''
from typing import List  # type hints for function signatures
def largestRectangleArea(heights: List[int]) -> int:
    """
    Find the largest rectangle area in a histogram.

    Args:
        heights: list of bar heights, each width=1
    Returns:
        maximum rectangle area

    Approach:
        Increasing monotonic stack of indices.
        Pop when current bar < stack top bar.
        width = i - stack[-1] - 1 (or i if stack empty).
        Append sentinel 0 to flush remaining bars.
    """
    stack = []            # for keeping indexes for a monotonic increasing stack of bars
                          # each element is a list of 2 [height, idx]
                          # the append-ed in the inserted pair is inherited from the index of the last evicted pair
    max_area = 0
    for i, h in enumerate (heights):
        cached_idx = i
        while stack and h < stack[-1][0]:
            popped_height, popped_idx = stack.pop()
            cached_idx = popped_idx   
            max_area = max(max_area, popped_height * (i - popped_idx))
        stack.append([h, cached_idx])
    while stack:
        h, idx = stack.pop()
        max_area = max(max_area , h * (len(heights) - idx))
        
    return max_area
            



# --- Debug prints (expected in comments) ---
print(largestRectangleArea([2,1,5,6,2,3]))  # 10
print(largestRectangleArea([2,4]))           # 4
print(largestRectangleArea([1]))             # 1
print(largestRectangleArea([6,6,6,6]))       # 24
print(largestRectangleArea([1,2,3,4,5]))     # 9
test_harness(largestRectangleArea)
r'''
10
4
1
24
9
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED
Test 6: PASSED

6/6 tests passed.
'''

r'''
def test_harness(func):
    """Run test cases for largestRectangleArea."""
    tests = [
        # (heights, expected_area)
        ([2,1,5,6,2,3], 10),
        ([2,4],          4),
        ([1],            1),
        ([0,0,0],        0),
        ([6,6,6,6],     24),
        ([1,2,3,4,5],   9),
    ]
    passed = 0
    for i, (heights, expected) in enumerate(tests):
        result = func(heights)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(f"Test {i+1}: {status}")
        if status == "FAILED":
            print(f"  Input:    {heights}")
            print(f"  Expected: {expected}")
            print(f"  Got:      {result}")
    print(f"\n{passed}/{len(tests)} tests passed.")
'''
pass

10
4
1
24
9
Test 1: PASSED
Test 2: PASSED
Test 3: PASSED
Test 4: PASSED
Test 5: PASSED
Test 6: PASSED

6/6 tests passed.


In [ ]:
# Uncomment and run when solution is ready
# test_harness(largestRectangleArea)

## Complexity

| Approach            | Time   | Space  |
|---------------------|--------|--------|
| Brute Force         | O(n²)  | O(1)   |
| Divide & Conquer    | O(n log n) | O(n) |
| Monotonic Stack     | O(n)   | O(n)   |

Each bar is pushed and popped at most once → O(n) amortized.

## Real World Connection

At Citi, dashboards aggregate time-series CPU metrics across 6,000
endpoints into histograms of utilization buckets.
Finding the largest rectangle is equivalent to finding the longest
continuous time window where all endpoints stayed below a threshold.
This drives capacity planning: if the rectangle is wide, the cluster
was consistently under-utilized and can be right-sized on AWS.
The O(n) stack approach processes a full day's histogram in one pass,
making it practical inside a CloudWatch Lambda trigger.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra